In [29]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np


In [30]:
### Load all the trained model,scaler pickle,onehot
model=load_model('model.h5')

### Load encoder and scaler 
with open('one_hot_encoder.pkl','rb') as file:
    one_hot_encoder=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

In [31]:
##Example input data 
input_data= {
    'CreditScore': 600,
    'Geography': 'Spain',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000

}

In [32]:
print(one_hot_encoder.categories_)


[array(['France', 'Germany', 'Spain'], dtype=object)]


In [33]:
geo_encoded= one_hot_encoder.transform([[input_data['Geography']]]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded ,columns=one_hot_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\hp\annclassification\annclassification\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,0.0,0.0,1.0


In [34]:
## combine  one-hot encoded columns with input data 
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Spain,Male,40,3,60000,2,1,1,50000


In [35]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,Spain,1,40,3,60000,2,1,1,50000


In [36]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,0.0,0.0,1.0


In [37]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.47252305,  0.91407657,  0.0906231 , -0.70903082, -0.27592043,
         0.79187296,  0.63599873,  0.97160303, -0.84394851, -1.01288298,
        -0.58166121,  1.77561931]])

In [38]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 [==============================] - 0s 211ms/step


array([[0.03647003]], dtype=float32)

In [39]:
prediction_proba = prediction[0][0]

In [40]:
prediction_proba

0.036470026

In [41]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
